# Re-implementation of Enhanced Resource Allocation index

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
from torch_geometric.datasets import KarateClub
from torch_geometric.utils import to_networkx, degree
from torch_sparse import SparseTensor
from sklearn.metrics.pairwise import cosine_similarity

import src.utils as ut
import src.train_utils as tr
import src.contrastive_pretrain as pretr
import src.contrastive_model as ctmod
import src.encoder as enc
import src.decoder as dec

hp = {
    'xdp': 0.7,
    'tdp': 0.3,
    'pt': 0.75,
    'gnnedp': 0.0,
    'preedp': 0.4,
    'predp': 0.05,
    'gnndp': 0.05,
    'res': False,
    'probscale': 4.3,
    'proboffset': 2.8,
    'alpha': 1.0,
    'gnnlr': 0.0043,
    'prelr': 0.0024,
    'batch_size': 1152,
    'ln': True,
    'lnnn': True,
    'epochs': 100,
    'model': 'puregcn',
    'runs': 1,
    'hiddim': 256,
    'mplayers': 1,
    'testbs': 8192,
    'maskinput': True,
    'jk': True,
    'use_xlin': True,
    'tailact': True,
    'use_valedges_as_input': False,
    'freeze': True,
    'inner': False,
    'ct_param': {
    	'learning_rate': 0.01,
    	'num_hidden': 256,
    	'num_proj_hidden': 32,
    	'activation': 'prelu',
    	'base_model': 'GCNConv',
    	'num_layers': 2,
    	'drop_edge_rate_1': 0.3,
    	'drop_edge_rate_2': 0.4,
    	'drop_feature_rate_1': 0.1,
    	'drop_feature_rate_2': 0.0,
    	'tau': 0.4,
    	'num_epochs': 1500,
    	'weight_decay': 1e-5,
    	'drop_scheme': 'degree',
    }
}

Note: to be able to use all crisp methods, you need to install some additional packages:  {'graph_tool', 'wurlitzer', 'infomap'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'ASLPAw'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'wurlitzer', 'infomap'}


In [2]:
def ERA(data, u, v, y, deg_out):
    adj1 = data.adj_t[u].storage.col()
    adj2 = data.adj_t[v].storage.col()
    overlaps = torch.unique(adj1[torch.isin(adj1, adj2)])# np.intersect1d(adj1, adj2)
    w_A = 0
    w_X = 0
    for cn in overlaps:
        w_A += 1/deg_out[cn]
        if y != 0:
            w_X += F.cosine_similarity(data.x[u].unsqueeze(0), data.x[cn].unsqueeze(0)) * F.cosine_similarity(data.x[cn].unsqueeze(0), data.x[v].unsqueeze(0))
    w_X += F.cosine_similarity(data.x[u].unsqueeze(0), data.x[v].unsqueeze(0))
    return (1-y)*w_A + y*w_X

In [21]:
class ERA_Mlp_Decoder(torch.nn.Module):
    """Hadamard-product-based MLP link predictor."""
    def __init__(self, embedding_size, hidden_size, data, total_steps):
        super().__init__()
        self.data = data
        self.degrees = degree(data.edge_index[0], num_nodes=data.num_nodes)
        self.y = 0.1
        self.embedding_size = embedding_size
        self.net = nn.Sequential(
            nn.Linear(embedding_size, hidden_size), nn.ReLU(), nn.Linear(hidden_size, 1)
        )
        self.total_steps = total_steps
        self.current_step = 0

    def update_y(self):
        if self.current_step < self.total_steps:
            self.y = 0.1 + (self.current_step / self.total_steps) * (1 - 0.1)
            self.current_step += 1
        print(self.y)
        
    def multidomainforward(
        self, x, adj, tar_ei, filled1: bool = False, cndropprobs: list[float] = []
    ):
        w_ERA = ERA(self.data, tar_ei[0], tar_ei[1], 0., self.degrees)
        x1 = x[tar_ei[0]]
        x2 = x[tar_ei[1]]
        prod = self.net(x1 * x2)
        res = w_ERA + 0.01*prod
        return torch.Tensor((0.5))

    def forward(self, x, adj, tar_ei, filled1: bool = False):
        mdforward = self.multidomainforward(x, adj, tar_ei)
        return torch.cat([torch.sigmoid(mdforward)], dim=-1)

class CosineDecayScheduler:
    def __init__(self, max_val, warmup_steps, total_steps):
        self.max_val = max_val
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps

    def get(self, step):
        if step < self.warmup_steps:
            return self.max_val * step / self.warmup_steps # augmentation de plus en plus grande
        elif self.warmup_steps <= step <= self.total_steps:
            return self.max_val * (1 + np.cos((step - self.warmup_steps) * np.pi /
                                              (self.total_steps - self.warmup_steps))) / 2 # décroit de façon lisse et progressive.
        else:
            raise ValueError('Step ({}) > total number of steps ({}).'.format(step, self.total_steps))

In [22]:
DATASET = 'Cora'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
evaluator = ut.get_evaluator(DATASET)
data_split = ut.DataSplit(DATASET, device, 1, True)
def init_model(data, hp):
    encoder = enc.ENCODER_NCN(data.num_features, hp['hiddim'], hp['hiddim'], hp['mplayers'],
                    hp['gnndp'], hp['ln'], hp['res'], data.max_x,
                    hp['model'], edrop=hp['gnnedp'],  xdropout=hp['xdp'], taildropout=hp['tdp']).to(device)
    predictor = ERA_Mlp_Decoder(hp['hiddim'], hp['hiddim'], data, hp['epochs']).to(device)
    return encoder, predictor
res = tr.runs('Cora GCN+ERA_MLP', init_model, None, data_split, evaluator, hp)

1 split from the dataset Cora


100%|██████████| 1/1 [00:00<00:00, 10.75it/s]


split time:  0.1  s
dataset split 
train edge 3696
valid edge 527
valid edge_neg 527
test edge 1055
test edge_neg 1054
### Cora GCN+ERA_MLP ###


  0%|          | 0/100 [00:00<?, ?it/s]

0.1


  0%|          | 0/100 [00:00<?, ?it/s]


TypeError: new(): data must be a sequence (got float)